In [1]:
import os

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"))

# Read credentials from the environment or local .env; never store keys in notebook source.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("GEMINI_API_KEY is not set; agent calls will safely route to MANUAL_REVIEW.")

In [2]:
import json
from datetime import datetime
from typing import List, Dict, Any
from collections import Counter

# 1. Travel Reimbursement Approval Agent

AI Developer Candidate Assignment

This notebook implements a lightweight Agentic AI prototype for evaluating
employee travel reimbursement claims against the provided travel reimbursement
policy.

The agent:
- grounds decisions in policy rules
- uses tools for deterministic policy checks
- uses an LLM to orchestrate the evaluation
- routes ambiguous, incomplete, high-value, or exception cases to Manual Review
- produces structured JSON results
- provides a minimal results dashboard

The evaluation is bounded to the five Appendix B claims. Deterministic
policy tools provide evidence; Gemini orchestrates those tools and explains
the outcome. Unresolved conditions are routed to manual review.

In [3]:
DECISIONS = {
    "APPROVE",
    "PARTIAL_APPROVE",
    "REJECT",
    "MANUAL_REVIEW",
}

In [4]:
OUTPUT_FIELDS = {
    "claim_id",
    "decision",
    "approved_amount",
    "deducted_amount",
    "missing_docs",
    "policy_refs",
    "confidence",
    "explanation",
    "tools_used"
}

## 2. Claim Intake and Input Validation

The five assignment-provided Appendix B claims are retained unchanged. Intake validation checks required claim structure before evaluation; JSON input can also be parsed through the helper below.

In [5]:
# These are the five assignment-provided Appendix B claims; do not add or alter cases.
claims = [
    {
        "claim_id": "CLM-001",
        "employee": "A. Rivera",
        "trip_start": "2026-06-10",
        "trip_end": "2026-06-12",
        "submitted": "2026-06-20",
        "expenses": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": 420.00,
                "receipt_attached": True,
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $180",
                "amount": 360.00,
                "receipt_attached": True,
            },
            {
                "category": "meals",
                "description": "Meals, 3 days @ ~$60/day",
                "amount": 180.00,
                "receipt_attached": True,
            },
            {
                "category": "conference_fees",
                "description": "Conference registration",
                "amount": 150.00,
                "receipt_attached": True,
            },
        ],
        "total_claimed": 1110.00,
    },

    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "trip_start": "2026-06-14",
        "trip_end": "2026-06-15",
        "submitted": "2026-06-25",
        "expenses": [
            {
                "category": "spa",
                "description": "Hotel spa package",
                "amount": 300.00,
                "receipt_attached": True,
            },
            {
                "category": "minibar",
                "description": "In-room minibar",
                "amount": 80.00,
                "receipt_attached": True,
            },
        ],
        "total_claimed": 380.00,
    },

    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "trip_start": "2026-06-08",
        "trip_end": "2026-06-10",
        "submitted": "2026-06-22",
        "expenses": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": 300.00,
                "receipt_attached": True,
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $250",
                "amount": 500.00,
                "receipt_attached": True,
            },
            {
                "category": "meals",
                "description": "Meals, 2 days @ $70/day",
                "amount": 140.00,
                "receipt_attached": True,
            },
        ],
        "total_claimed": 940.00,
    },

    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "trip_start": "2026-06-16",
        "trip_end": "2026-06-18",
        "submitted": "2026-06-28",
        "expenses": [
            {
                "category": "airfare",
                "description": "Business-class international airfare",
                "amount": 2400.00,
                "receipt_attached": True,
                "airfare_class": "business",
            },
            {
                "category": "lodging",
                "description": "Hotel, 3 nights",
                "amount": 600.00,
                "receipt_attached": False,
            },
        ],
        "total_claimed": 3000.00,
    },

    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "trip_start": "2026-06-11",
        "trip_end": "2026-06-11",
        "submitted": "2026-06-24",
        "expenses": [
            {
                "category": "meals",
                "description": "Client dinner for 4 (business development)",
                "amount": 220.00,
                "receipt_attached": False,
            },
        ],
        "total_claimed": 220.00,
    },
]

## 3. Policy Context

The following policy is derived from Appendix A of the assignment.

Each rule retains its stable `POL-*` identifier so that agent decisions
can be traced back to the applicable policy.

In [6]:
POLICY = {
    "categories": {
        "eligible": [
            "airfare",
            "lodging",
            "meals",
            "ground_transport",
            "conference_fees",
        ],
        "ineligible": [
            "alcohol",
            "minibar",
            "spa",
            "gym",
            "personal_entertainment",
            "in_room_movies",
            "personal_shopping",
            "gifts",
            "traffic_fines",
            "penalties",
            "late_fees",
            "personal",
        ],
        "eligible_policy_ref": "POL-CAT-01",
        "ineligible_policy_ref": "POL-CAT-02",
    },

    "limits": {
        "meals": {
            "max_per_day": 75.00,
            "policy_ref": "POL-PD-01",
        },
        "lodging": {
            "max_per_night": 200.00,
            "policy_ref": "POL-PD-02",
        },
        "ground_transport": {
            "max_per_day": 50.00,
            "policy_ref": "POL-PD-03",
        },
    },

    "airfare": {
        "allowed_classes": ["economy"],
        "policy_ref": "POL-AIR-01",
        "exception_action": "MANUAL_REVIEW",
    },

    "receipts": {
        "threshold": 25.00,
        "always_required": [
            "airfare",
            "lodging",
        ],
        "require_itemized_receipt": True,
        "policy_ref": "POL-RCT-01",
        "missing_receipt_action": "MANUAL_REVIEW",
        "missing_receipt_policy_ref": "POL-RCT-02",
    },

    "approval": {
        "auto_approve_max": 500.00,
        "manager_max": 2000.00,
        "above_manager_max_action": "MANUAL_REVIEW",
        "auto_approve_policy_ref": "POL-APR-01",
        "manager_policy_ref": "POL-APR-02",
        "director_policy_ref": "POL-APR-03",
    },

    "timeliness": {
        "submission_window_days": 30,
        "late_claim_action": "MANUAL_REVIEW",
        "policy_ref": "POL-TIME-01",
    },
}

## 4A — Policy Assertions

These checks catch inconsistencies in the policy configuration before the agent uses it.

In [7]:
def assert_policy():
    assert "airfare" in POLICY["categories"]["eligible"]
    assert "lodging" in POLICY["categories"]["eligible"]
    assert "meals" in POLICY["categories"]["eligible"]

    assert "spa" in POLICY["categories"]["ineligible"]
    assert "minibar" in POLICY["categories"]["ineligible"]

    assert POLICY["limits"]["meals"]["max_per_day"] == 75.00
    assert POLICY["limits"]["lodging"]["max_per_night"] == 200.00
    assert POLICY["limits"]["ground_transport"]["max_per_day"] == 50.00

    assert POLICY["airfare"]["allowed_classes"] == ["economy"]

    assert POLICY["receipts"]["threshold"] == 25.00
    assert "airfare" in POLICY["receipts"]["always_required"]
    assert "lodging" in POLICY["receipts"]["always_required"]

    assert POLICY["approval"]["auto_approve_max"] == 500.00
    assert POLICY["approval"]["manager_max"] == 2000.00

    assert POLICY["timeliness"]["submission_window_days"] == 30

    print("Policy validation passed.")

In [8]:
assert_policy()

Policy validation passed.


## 4B — Tool 1: Expense Eligibility Checker

Checks whether an expense category is explicitly eligible or ineligible
under the travel reimbursement policy.

The tool does not make the final claim-level decision. It returns
policy-grounded evidence that the agent can use.

In [9]:
def check_expense_eligibility(
    category: str,
    description: str
) -> Dict[str, Any]:

    category = category.strip().lower()

    if category in POLICY["categories"]["ineligible"]:
        return {
            "category": category,
            "eligible": False,
            "action": "REJECT",
            "policy_ref": POLICY["categories"]["ineligible_policy_ref"],
            "reason": (
                f"'{category}' is explicitly listed as an "
                "ineligible expense category."
            ),
        }

    if category in POLICY["categories"]["eligible"]:
        return {
            "category": category,
            "eligible": True,
            "action": "CONTINUE",
            "policy_ref": POLICY["categories"]["eligible_policy_ref"],
            "reason": (
                f"'{category}' is an eligible reimbursement category "
                "when incurred for a documented business purpose."
            ),
        }

    return {
        "category": category,
        "eligible": None,
        "action": "MANUAL_REVIEW",
        "policy_ref": None,
        "reason": (
            f"Category '{category}' is not explicitly covered by "
            "the provided eligible or ineligible category lists."
        ),
    }

In [10]:
test_eligibility_1 = check_expense_eligibility(
    category="meals",
    description="Client dinner"
)

test_eligibility_2 = check_expense_eligibility(
    category="spa",
    description="Hotel spa package"
)

test_eligibility_3 = check_expense_eligibility(
    category="unknown_category",
    description="Something unusual"
)

print(test_eligibility_1)
print(test_eligibility_2)
print(test_eligibility_3)

{'category': 'meals', 'eligible': True, 'action': 'CONTINUE', 'policy_ref': 'POL-CAT-01', 'reason': "'meals' is an eligible reimbursement category when incurred for a documented business purpose."}
{'category': 'spa', 'eligible': False, 'action': 'REJECT', 'policy_ref': 'POL-CAT-02', 'reason': "'spa' is explicitly listed as an ineligible expense category."}
{'category': 'unknown_category', 'eligible': None, 'action': 'MANUAL_REVIEW', 'policy_ref': None, 'reason': "Category 'unknown_category' is not explicitly covered by the provided eligible or ineligible category lists."}


## 4C — Tool 2: Receipt Requirement Checker

Determines whether an expense requires a receipt and whether the
required receipt has been provided.

In [11]:
def check_receipt_requirement(
    category: str,
    amount: float,
    receipt_attached: bool
) -> Dict[str, Any]:

    category = category.strip().lower()

    always_required = (
        category in POLICY["receipts"]["always_required"]
    )

    above_threshold = (
        amount > POLICY["receipts"]["threshold"]
    )

    receipt_required = always_required or above_threshold

    if not receipt_required:
        return {
            "category": category,
            "amount": amount,
            "receipt_required": False,
            "receipt_attached": receipt_attached,
            "action": "CONTINUE",
            "policy_ref": POLICY["receipts"]["policy_ref"],
            "reason": "No receipt is required under the policy.",
        }

    if receipt_attached:
        return {
            "category": category,
            "amount": amount,
            "receipt_required": True,
            "receipt_attached": True,
            "action": "CONTINUE",
            "policy_ref": POLICY["receipts"]["policy_ref"],
            "reason": "Required receipt is attached.",
        }

    return {
        "category": category,
        "amount": amount,
        "receipt_required": True,
        "receipt_attached": False,
        "action": "MANUAL_REVIEW",
        "policy_ref": POLICY["receipts"]["missing_receipt_policy_ref"],
        "reason": (
            "A required receipt is missing. "
            "The claim must be routed to Manual Review."
        ),
    }

In [12]:
# CLM-004 lodging
result_004 = check_receipt_requirement(
    category="lodging",
    amount=600.00,
    receipt_attached=False
)

# CLM-005 meals
result_005 = check_receipt_requirement(
    category="meals",
    amount=220.00,
    receipt_attached=False
)

print("CLM-004 lodging:")
print(result_004)

print("\nCLM-005 meals:")
print(result_005)

CLM-004 lodging:
{'category': 'lodging', 'amount': 600.0, 'receipt_required': True, 'receipt_attached': False, 'action': 'MANUAL_REVIEW', 'policy_ref': 'POL-RCT-02', 'reason': 'A required receipt is missing. The claim must be routed to Manual Review.'}

CLM-005 meals:
{'category': 'meals', 'amount': 220.0, 'receipt_required': True, 'receipt_attached': False, 'action': 'MANUAL_REVIEW', 'policy_ref': 'POL-RCT-02', 'reason': 'A required receipt is missing. The claim must be routed to Manual Review.'}


## 4D — Tool 3: Reimbursement Calculator

Calculates the reimbursable and deducted amount for an individual
expense according to the applicable policy limit.

The tool performs deterministic calculations. It does not make the
claim-level approval decision.

In [13]:
def calculate_reimbursement(
    category: str,
    amount: float,
    units: float = 1
) -> Dict[str, Any]:

    category = category.strip().lower()

    # Categories with no per-unit cap
    if category not in POLICY["limits"]:
        return {
            "category": category,
            "claimed_amount": amount,
            "reimbursable_amount": amount,
            "deducted_amount": 0.0,
            "limit_applied": False,
            "policy_ref": None,
            "action": "CONTINUE",
            "reason": "No per-diem limit applies to this category.",
        }

    if units <= 0:
        return {
            "category": category,
            "claimed_amount": amount,
            "reimbursable_amount": 0.0,
            "deducted_amount": amount,
            "limit_applied": False,
            "policy_ref": POLICY["limits"][category]["policy_ref"],
            "action": "MANUAL_REVIEW",
            "reason": "Expense units must be greater than zero.",
        }

    if category == "meals":
        unit_limit = POLICY["limits"]["meals"]["max_per_day"]
        unit_type = "day"

    elif category == "lodging":
        unit_limit = POLICY["limits"]["lodging"]["max_per_night"]
        unit_type = "night"

    elif category == "ground_transport":
        unit_limit = POLICY["limits"]["ground_transport"]["max_per_day"]
        unit_type = "day"

    else:
        return {
            "category": category,
            "claimed_amount": amount,
            "reimbursable_amount": amount,
            "deducted_amount": 0.0,
            "limit_applied": False,
            "policy_ref": None,
            "action": "CONTINUE",
            "reason": "No applicable reimbursement limit found.",
        }

    maximum_reimbursable = unit_limit * units

    reimbursable_amount = min(
        amount,
        maximum_reimbursable
    )

    deducted_amount = amount - reimbursable_amount

    return {
        "category": category,
        "claimed_amount": round(amount, 2),
        "units": units,
        "unit_type": unit_type,
        "unit_limit": unit_limit,
        "maximum_reimbursable": round(maximum_reimbursable, 2),
        "reimbursable_amount": round(reimbursable_amount, 2),
        "deducted_amount": round(deducted_amount, 2),
        "limit_applied": True,
        "policy_ref": POLICY["limits"][category]["policy_ref"],
        "action": (
            "PARTIAL_REIMBURSEMENT"
            if deducted_amount > 0
            else "CONTINUE"
        ),
    }

## 4E — Tool 4: Additional Claim-Level Policy Checks

Airfare class, submission timeliness, and approval authority apply to the trip or claim total rather than to an individual expense, so they are checked separately.

In [14]:
def check_airfare_class(airfare_class: str) -> Dict[str, Any]:
    if airfare_class.lower() == "economy":
        return {
            "allowed": True,
            "action": "CONTINUE",
            "policy_ref": "POL-AIR-01",
        }

    return {
        "allowed": False,
        "action": "MANUAL_REVIEW",
        "policy_ref": "POL-AIR-01",
        "reason": (
            f"{airfare_class} class airfare requires manual review "
            "because economy class is the standard allowed class."
        ),
    }

# 4F - Tool 5: Validate submission timelines

In [15]:
def check_submission_timeliness(
    travel_end_date: str,
    submission_date: str,
) -> Dict[str, Any]:
    travel_end = datetime.strptime(travel_end_date, "%Y-%m-%d")
    submitted = datetime.strptime(submission_date, "%Y-%m-%d")

    days_after_trip = (submitted - travel_end).days

    if days_after_trip <= POLICY["timeliness"]["submission_window_days"]:
        return {
            "within_window": True,
            "action": "CONTINUE",
            "days_after_trip": days_after_trip,
            "policy_ref": "POL-TIME-01",
        }

    return {
        "within_window": False,
        "action": "MANUAL_REVIEW",
        "days_after_trip": days_after_trip,
        "policy_ref": "POL-TIME-01",
        "reason": (
            f"Claim was submitted {days_after_trip} days after "
            "the end of travel, exceeding the 30-day submission window."
        ),
    }

# 4G - Tool 6: Checking Approval Threshold

In [16]:
def check_approval_threshold(
    reimbursable_amount: float
) -> Dict[str, Any]:

    if reimbursable_amount <= POLICY["approval"]["auto_approve_max"]:
        return {
            "reimbursable_amount": round(reimbursable_amount, 2),
            "tier": "AUTO_APPROVE",
            "action": "APPROVE",
            "policy_ref": POLICY["approval"]["auto_approve_policy_ref"],
            "reason": (
                "Reimbursable amount is within the automatic "
                "approval threshold."
            ),
        }

    if reimbursable_amount <= POLICY["approval"]["manager_max"]:
        return {
            "reimbursable_amount": round(reimbursable_amount, 2),
            "tier": "MANAGER",
            "action": "APPROVE",
            "policy_ref": POLICY["approval"]["manager_policy_ref"],
            "reason": (
                "Reimbursable amount exceeds the automatic approval "
                "threshold but is within the manager approval limit."
            ),
        }

    return {
        "reimbursable_amount": round(reimbursable_amount, 2),
        "tier": "DIRECTOR",
        "action": "MANUAL_REVIEW",
        "policy_ref": POLICY["approval"]["director_policy_ref"],
        "reason": (
            "Reimbursable amount exceeds the agent's approval authority "
            "and requires Manual Review."
        ),
    }

## 4H — Gemini Tool Declarations

These schemas describe the Python tools to Gemini: their names, purposes, and arguments. The schemas make functions selectable by the model; Python implementations still perform the checks.

In [17]:
GEMINI_TOOLS = [
    {
        "type": "function",
        "name": "check_expense_eligibility",
        "description": (
            "Check whether a reimbursement expense category is "
            "explicitly eligible or ineligible under the travel "
            "reimbursement policy."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Expense category."
                },
                "description": {
                    "type": "string",
                    "description": "Description of the expense."
                }
            },
            "required": ["category", "description"]
        }
    },
    {
        "type": "function",
        "name": "check_receipt_requirement",
        "description": (
            "Check whether a receipt is required for an expense "
            "and whether the required receipt is attached."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Expense category."
                },
                "amount": {
                    "type": "number",
                    "description": "Claimed expense amount."
                },
                "receipt_attached": {
                    "type": "boolean",
                    "description": "Whether a receipt is attached."
                }
            },
            "required": [
                "category",
                "amount",
                "receipt_attached"
            ]
        }
    },
    {
        "type": "function",
        "name": "calculate_reimbursement",
        "description": (
            "Calculate the reimbursable amount for an expense after "
            "applying applicable policy limits."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Expense category."
                },
                "amount": {
                    "type": "number",
                    "description": "Claimed expense amount."
                },
                "units": {
                    "type": "number",
                    "description": (
                        "Number of applicable days or nights."
                    )
                }
            },
            "required": [
                "category",
                "amount",
                "units"
            ]
        }
    },
    {
        "type": "function",
        "name": "check_approval_threshold",
        "description": (
            "Determine the applicable approval authority based on "
            "the total reimbursable amount."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "reimbursable_amount": {
                    "type": "number",
                    "description": (
                        "Total reimbursable amount after deductions."
                    )
                }
            },
            "required": ["reimbursable_amount"]
        }
    },
    {
    "type": "function",
    "name": "check_airfare_class",
    "description": (
        "Checks whether the airfare class complies with the travel "
        "reimbursement policy. Economy is allowed. Non-economy airfare "
        "requires MANUAL_REVIEW."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "airfare_class": {
                "type": "string",
                "description": "Airfare class, such as economy, business, or first."
            }
        },
        "required": ["airfare_class"],
    },
},
{
    "type": "function",
    "name": "check_submission_timeliness",
    "description": (
        "Checks whether a reimbursement claim was submitted within "
        "the policy's allowed submission window after travel."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "travel_end_date": {
                "type": "string",
                "description": "Travel end date in YYYY-MM-DD format."
            },
            "submission_date": {
                "type": "string",
                "description": "Claim submission date in YYYY-MM-DD format."
            }
        },
        "required": [
            "travel_end_date",
            "submission_date"
        ],
    },
},
]

print("Gemini tools configured:")
for tool in GEMINI_TOOLS:
    print("-", tool["name"])

Gemini tools configured:
- check_expense_eligibility
- check_receipt_requirement
- calculate_reimbursement
- check_approval_threshold
- check_airfare_class
- check_submission_timeliness


In [18]:
from google.genai import Client

gemini_client = (
    Client(api_key=GEMINI_API_KEY)
    if GEMINI_API_KEY
    else None
)

print("Gemini client initialized." if gemini_client else "Gemini client unavailable; safe fallback enabled.")

Gemini client initialized.


## 4I — Tool Dispatch

This dispatcher maps the function name returned by Gemini to its local Python implementation. Gemini requests the tool; the dispatcher executes it and returns the result to the agent loop.

In [19]:
def execute_tool(
    tool_name: str,
    tool_args: Dict[str, Any]
) -> Dict[str, Any]:

    if tool_name == "check_airfare_class":
        return check_airfare_class(
            airfare_class=tool_args["airfare_class"],
        )

    if tool_name == "check_submission_timeliness":
        return check_submission_timeliness(
            travel_end_date=tool_args["travel_end_date"],
            submission_date=tool_args["submission_date"],
        )

    if tool_name == "check_expense_eligibility":
        return check_expense_eligibility(
            category=tool_args["category"],
            description=tool_args["description"],
        )

    if tool_name == "check_receipt_requirement":
        return check_receipt_requirement(
            category=tool_args["category"],
            amount=tool_args["amount"],
            receipt_attached=tool_args["receipt_attached"],
        )

    if tool_name == "calculate_reimbursement":
        return calculate_reimbursement(
            category=tool_args["category"],
            amount=tool_args["amount"],
            units=tool_args["units"],
        )

    if tool_name == "check_approval_threshold":
        return check_approval_threshold(
            reimbursable_amount=tool_args["reimbursable_amount"],
        )

    raise ValueError(f"Unknown tool: {tool_name}")

## 5 — Agent Instructions

The Gemini model acts as a Travel Reimbursement Approval Agent.

The agent must:
1. Analyze the submitted claim and its individual expenses.
2. Use the available policy tools whenever policy verification or calculation is required.
3. Never invent policy references.
4. Use tool results as the authoritative source for eligibility, receipt requirements, reimbursement limits, and approval thresholds.
5. Route ambiguous, incomplete, exceptional, or otherwise unresolved cases to MANUAL_REVIEW.
6. Produce exactly one of:
   - APPROVE
   - PARTIAL_APPROVE
   - REJECT
   - MANUAL_REVIEW
7. Return the required reimbursement output fields.

In [20]:
AGENT_SYSTEM_PROMPT = """
You are a Travel Reimbursement Approval Agent.

Evaluate the provided travel reimbursement claim using the reimbursement
policy and the available deterministic policy tools.

Your final response must be exactly one JSON object with these fields:

{
  "claim_id": "...",
  "decision": "APPROVE | PARTIAL_APPROVE | REJECT | MANUAL_REVIEW",
  "approved_amount": 0.0,
  "deducted_amount": 0.0,
  "missing_docs": [],
  "policy_refs": [],
  "confidence": 0.0,
  "explanation": "...",
  "tools_used": []
}

EXECUTION STRATEGY

Use at most two tool-calling rounds.

ROUND 1 — COLLECT ALL INDEPENDENT EVIDENCE

First inspect the ENTIRE claim.

Identify every expense in the claim before making any tool calls.

Then request ALL independent checks that can be determined directly
from the original claim in the SAME tool-calling response.

For EVERY expense, request these three tools together:

1. check_expense_eligibility
2. check_receipt_requirement
3. calculate_reimbursement

For the claim itself:

4. check_submission_timeliness

If the claim contains airfare:

5. check_airfare_class

Do NOT wait for one tool result before requesting another independent
check.

Do NOT request only one expense at a time.

Do NOT defer independent expense checks to Round 2.

The calculate_reimbursement call must use:
- lodging: number of nights
- meals: number of applicable days
- ground_transport: number of applicable days
- other categories: units = 1 unless the claim clearly specifies
  another applicable quantity.

ROUND 2 — FINALIZE THE DECISION

After receiving the Round 1 tool results:

1. Determine the total reimbursable amount.
2. Determine the total deductions.
3. Identify any missing required documents.
4. Determine whether any Manual Review condition exists.
5. ALWAYS call check_approval_threshold using the total
   reimbursable amount, even if another Manual Review condition
   has already been identified.

The approval threshold check is the required Round 2 dependency
whenever expense-level reimbursement results are available.

Round 2 should contain ONLY tool calls that depend on Round 1 results,
followed by the final JSON response.

Do NOT repeat any expense-level tool call from Round 1.

Do NOT call eligibility, receipt, reimbursement, airfare-class, or
submission-timeliness tools again in Round 2 unless the original claim
contained genuinely missing information that made the Round 1 check
impossible.

DECISION RULES

APPROVE:
The entire claim is reimbursable, all required evidence is present,
and no unresolved policy issue exists.

PARTIAL_APPROVE:
Some amount is reimbursable and another amount is deducted because of
an applicable reimbursement limit or other policy deduction, with no
Manual Review condition.

REJECT:
The expenses are not reimbursable under the applicable policy.

MANUAL_REVIEW:
The claim cannot be finalized automatically because of missing required
evidence, ambiguity, a policy exception, non-economy airfare, a late
submission, or an approval condition requiring review.

POLICY AND TOOL AUTHORITY

Treat deterministic tool results as authoritative.

Do not invent policy rules.

Do not invent policy references.

Every policy reference in the final result must come from the provided
policy or a tool result.

An explicitly ineligible expense must not be reimbursed.

A missing required receipt requires MANUAL_REVIEW.

Non-economy airfare requires MANUAL_REVIEW when indicated by the
airfare-class tool.

A claim submitted outside the allowed submission window requires
MANUAL_REVIEW when indicated by the timeliness tool.

APPROVED AND DEDUCTED AMOUNTS

approved_amount must represent the amount actually approved by the
final decision.

For MANUAL_REVIEW, do not treat a potentially reimbursable amount as
already approved.

deducted_amount must represent amounts actually deducted under an
applicable policy rule.

MISSING DOCUMENTS

missing_docs must contain descriptions of required documents that are
actually absent.

CONFIDENCE

confidence must be a number between 0 and 1 and should reflect how
certain the agent is about the final decision based on the available
evidence.

TOOLS_USED

tools_used must contain the names of tools actually invoked.

Return ONLY the final JSON object when evaluation is complete.
"""

# 6 - Declaration of gemini tools for gemini APK

In [21]:
from google.genai import types

gemini_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name=tool["name"],
            description=tool["description"],
            parameters=tool["parameters"],
        )
        for tool in GEMINI_TOOLS
    ]
)

print("Gemini function declarations created.")

Gemini function declarations created.


# 7 - Gemini Quota Error Handling

In [22]:
def is_gemini_quota_error(exc: Exception) -> bool:
    error_text = str(exc).lower()

    return (
        "resource_exhausted" in error_text
        or "quota" in error_text
        or "429" in error_text
        or "too many requests" in error_text
    )

# 8 - Master run pipeline

In [23]:
import time


def run_reimbursement_agent(
    claim: Dict[str, Any],
    model: str = "gemini-3.5-flash-lite",
) -> Dict[str, Any]:

    if gemini_client is None:
        return {
            "claim_id": claim["claim_id"],
            "decision": "MANUAL_REVIEW",
            "approved_amount": 0.0,
            "deducted_amount": 0.0,
            "missing_docs": [],
            "policy_refs": [],
            "confidence": 0.0,
            "explanation": (
                "Gemini evaluation was not run because GEMINI_API_KEY is unset. "
                "Configure the key and rerun the agent cells for a policy evaluation."
            ),
            "tools_used": [],
        }

    claim_json = json.dumps(claim, indent=2)

    user_prompt = f"""
Evaluate the following travel reimbursement claim.

CLAIM:
{claim_json}

Follow the reimbursement policy and use the available tools
whenever required.

Complete the evaluation and return the final JSON object.
"""

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=user_prompt)
            ],
        )
    ]

    tools_used = []
    executed_tool_calls = set()

    thinking_config = types.ThinkingConfig(
        thinking_level="low"
    )

    MAX_GEMINI_ROUNDS = 2
    round_timings = []

    for round_number in range(1, MAX_GEMINI_ROUNDS + 1):

        round_start = time.perf_counter()

        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=AGENT_SYSTEM_PROMPT,
                    tools=[gemini_tool],
                    temperature=0.0,
                    thinking_config=thinking_config,
                ),
            )

        except Exception as exc:
            round_elapsed = time.perf_counter() - round_start

            if is_gemini_quota_error(exc):
                print(
                    f"Gemini quota/rate limit reached "
                    f"during round {round_number} "
                    f"after {round_elapsed:.2f}s."
                )

                return {
                    "claim_id": claim["claim_id"],
                    "decision": "MANUAL_REVIEW",
                    "approved_amount": 0.0,
                    "deducted_amount": 0.0,
                    "missing_docs": [],
                    "policy_refs": [],
                    "confidence": 0.0,
                    "explanation": (
                        "Automated evaluation could not be completed because "
                        "the Gemini API quota or rate limit was reached."
                    ),
                    "tools_used": list(dict.fromkeys(tools_used)),
                }

            raise

        round_elapsed = time.perf_counter() - round_start
        round_timings.append(round_elapsed)

        usage = response.usage_metadata

        print(
            f"Round {round_number}: "
            f"{round_elapsed:.2f}s | "
            f"prompt={getattr(usage, 'prompt_token_count', None)} | "
            f"candidates={getattr(usage, 'candidates_token_count', None)} | "
            f"thoughts={getattr(usage, 'thoughts_token_count', None)} | "
            f"total={getattr(usage, 'total_token_count', None)}"
        )

        function_calls = []

        for candidate in response.candidates:
            for part in candidate.content.parts:
                if part.function_call:
                    function_calls.append(part.function_call)

        # Gemini returned a final answer without requesting any more tools.
        if not function_calls:
            break

        # Preserve Gemini's response in the conversation.
        contents.append(response.candidates[0].content)

        tool_response_parts = []

        print(
            f"Round {round_number}: "
            f"{len(function_calls)} function call(s)"
        )

        for function_call in function_calls:

            tool_name = function_call.name
            tool_args = dict(function_call.args)

            tool_signature = (
                tool_name,
                json.dumps(tool_args, sort_keys=True),
            )

            # Prevent duplicate tool execution.
            if tool_signature in executed_tool_calls:
                continue

            executed_tool_calls.add(tool_signature)

            tool_start = time.perf_counter()

            result = execute_tool(
                tool_name,
                tool_args,
            )

            tool_elapsed = time.perf_counter() - tool_start

            print(
                f"  Tool: {tool_name} | "
                f"{tool_elapsed:.4f}s"
            )

            tools_used.append(tool_name)

            tool_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response=result,
                )
            )

        # Add tool results back to the conversation.
        contents.append(
            types.Content(
                role="user",
                parts=tool_response_parts,
            )
        )

    # ------------------------------------------------------------------
    # FINALIZATION
    #
    # If the final tool-calling round has completed, ask Gemini to
    # synthesize the final JSON WITHOUT giving it access to tools.
    # ------------------------------------------------------------------

    if round_number == MAX_GEMINI_ROUNDS:

        final_start = time.perf_counter()

        try:
            final_response = gemini_client.models.generate_content(
                model=model,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=AGENT_SYSTEM_PROMPT,
                    response_mime_type="application/json",
                    thinking_config=thinking_config,
                ),
            )

        except Exception as exc:
            final_elapsed = time.perf_counter() - final_start

            if is_gemini_quota_error(exc):
                print(
                    "Gemini quota/rate limit reached during finalization "
                    f"after {final_elapsed:.2f}s."
                )

                return {
                    "claim_id": claim["claim_id"],
                    "decision": "MANUAL_REVIEW",
                    "approved_amount": 0.0,
                    "deducted_amount": 0.0,
                    "missing_docs": [],
                    "policy_refs": [],
                    "confidence": 0.0,
                    "explanation": (
                        "Automated evaluation could not be completed because "
                        "the Gemini API quota or rate limit was reached "
                        "during finalization."
                    ),
                    "tools_used": list(dict.fromkeys(tools_used)),
                }

            raise

        final_elapsed = time.perf_counter() - final_start

        print(
            f"Finalization: "
            f"{final_elapsed:.2f}s"
        )

        final_text = final_response.text.strip()

    else:
        # Gemini produced the final answer within the normal tool loop.
        final_text = response.text.strip()

    # Remove accidental markdown code fences if Gemini adds them.
    if final_text.startswith("```"):
        final_text = final_text.replace("```json", "")
        final_text = final_text.replace("```", "")
        final_text = final_text.strip()

    result = json.loads(final_text)

    # Ensure tools_used reflects the tools actually executed.
    result["tools_used"] = list(dict.fromkeys(tools_used))

    print(
        f"Agent completed in {len(round_timings)} Gemini tool-calling "
        f"round(s). "
        f"Total Gemini tool-calling time: "
        f"{sum(round_timings):.2f}s"
    )

    print(
        f"Tools used: {list(dict.fromkeys(tools_used))}"
    )

    return result

In [24]:
result_001 = run_reimbursement_agent(claims[0])
result_001

Round 1: 2.57s | prompt=1906 | candidates=440 | thoughts=None | total=2346
Round 1: 14 function call(s)
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_submission_timeliness | 0.0000s
  Tool: check_airfare_class | 0.0000s


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Round 2: 1.31s | prompt=3263 | candidates=24 | thoughts=None | total=3287
Round 2: 1 function call(s)
  Tool: check_approval_threshold | 0.0000s
Finalization: 1.86s
Agent completed in 2 Gemini tool-calling round(s). Total Gemini tool-calling time: 3.88s
Tools used: ['check_expense_eligibility', 'check_receipt_requirement', 'calculate_reimbursement', 'check_submission_timeliness', 'check_airfare_class', 'check_approval_threshold']


{'claim_id': 'CLM-001',
 'decision': 'APPROVE',
 'approved_amount': 1110.0,
 'deducted_amount': 0.0,
 'missing_docs': [],
 'policy_refs': ['POL-CAT-01',
  'POL-RCT-01',
  'POL-PD-02',
  'POL-PD-01',
  'POL-TIME-01',
  'POL-AIR-01',
  'POL-APR-02'],
 'confidence': 1.0,
 'explanation': 'All expenses are eligible, receipts are attached, submission is timely, airfare is economy, and the amount has been verified and approved under manager approval threshold policy.',
 'tools_used': ['check_expense_eligibility',
  'check_receipt_requirement',
  'calculate_reimbursement',
  'check_submission_timeliness',
  'check_airfare_class',
  'check_approval_threshold']}

In [25]:
result_002 = run_reimbursement_agent(claims[1])
result_002

Round 1: 1.52s | prompt=1799 | candidates=222 | thoughts=None | total=2021
Round 1: 7 function call(s)
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_submission_timeliness | 0.0000s
Round 2: 1.33s | prompt=2444 | candidates=21 | thoughts=None | total=2465
Round 2: 1 function call(s)
  Tool: check_approval_threshold | 0.0000s
Finalization: 1.74s
Agent completed in 2 Gemini tool-calling round(s). Total Gemini tool-calling time: 2.86s
Tools used: ['check_expense_eligibility', 'check_receipt_requirement', 'calculate_reimbursement', 'check_submission_timeliness', 'check_approval_threshold']


{'claim_id': 'CLM-002',
 'decision': 'REJECT',
 'approved_amount': 0.0,
 'deducted_amount': 380.0,
 'missing_docs': [],
 'policy_refs': ['POL-CAT-02', 'POL-RCT-01', 'POL-TIME-01', 'POL-APR-01'],
 'confidence': 1.0,
 'explanation': 'All expenses in the claim (spa package and in-room minibar) are explicitly classified as ineligible under policy POL-CAT-02. Therefore, the claim is rejected.',
 'tools_used': ['check_expense_eligibility',
  'check_receipt_requirement',
  'calculate_reimbursement',
  'check_submission_timeliness',
  'check_approval_threshold']}

## 9 — Systematic Execution Logging

For each claim, the agent runner prints Gemini round duration and token counts, the number of requested function calls, each tool name and execution duration, finalization time, and the tools actually used. Quota failures are reported before the safe manual-review fallback. These are transient notebook output logs, not a persisted audit file; they help trace a run but do not expose Gemini's private internal reasoning.

In [26]:
result_003 = run_reimbursement_agent(claims[2])
result_003

Round 1: 1.63s | prompt=1860 | candidates=350 | thoughts=None | total=2210
Round 1: 11 function call(s)
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_submission_timeliness | 0.0000s
  Tool: check_airfare_class | 0.0000s
Round 2: 1.23s | prompt=2935 | candidates=23 | thoughts=None | total=2958
Round 2: 1 function call(s)
  Tool: check_approval_threshold | 0.0000s
Finalization: 2.15s
Agent completed in 2 Gemini tool-calling round(s). Total Gemini tool-calling time: 2.86s
Tools used: ['check_expense_eligibility', 'check_receipt_requirement', 'calculate_reimbursement', 'check_submission_timeliness', 'check_airfare_class', 'check_approval_

{'claim_id': 'CLM-003',
 'decision': 'PARTIAL_APPROVE',
 'approved_amount': 840.0,
 'deducted_amount': 100.0,
 'missing_docs': [],
 'policy_refs': ['POL-CAT-01',
  'POL-RCT-01',
  'POL-PD-02',
  'POL-PD-01',
  'POL-TIME-01',
  'POL-AIR-01',
  'POL-APR-02'],
 'confidence': 1.0,
 'explanation': 'All expenses are eligible, timely submitted, and have required receipts attached. Airfare and meals are fully reimbursable. Lodging exceeded the per-night limit of $200 (claimed $250/night for 2 nights, capped at $400 total), resulting in a $100 deduction. The final reimbursable amount of $840 requires manager approval per POL-APR-02, which is part of normal processing.',
 'tools_used': ['check_expense_eligibility',
  'check_receipt_requirement',
  'calculate_reimbursement',
  'check_submission_timeliness',
  'check_airfare_class',
  'check_approval_threshold']}

In [27]:
result_004 = run_reimbursement_agent(claims[3])
result_004

Round 1: 2.35s | prompt=1817 | candidates=254 | thoughts=373 | total=2444
Round 1: 8 function call(s)
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_airfare_class | 0.0000s
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_submission_timeliness | 0.0000s
Round 2: 1.85s | prompt=2981 | candidates=24 | thoughts=200 | total=3205
Round 2: 1 function call(s)
  Tool: check_approval_threshold | 0.0000s
Finalization: 2.56s
Agent completed in 2 Gemini tool-calling round(s). Total Gemini tool-calling time: 4.19s
Tools used: ['check_expense_eligibility', 'check_receipt_requirement', 'calculate_reimbursement', 'check_airfare_class', 'check_submission_timeliness', 'check_approval_threshold']


{'claim_id': 'CLM-004',
 'decision': 'MANUAL_REVIEW',
 'approved_amount': 0.0,
 'deducted_amount': 0.0,
 'missing_docs': ['lodging receipt'],
 'policy_refs': ['POL-CAT-01',
  'POL-RCT-01',
  'POL-AIR-01',
  'POL-RCT-02',
  'POL-PD-02',
  'POL-TIME-01',
  'POL-APR-03'],
 'confidence': 1.0,
 'explanation': 'The claim requires manual review due to multiple policy conditions: business-class airfare requires manual review, the lodging receipt is missing, and the total reimbursable amount exceeds the automated approval threshold.',
 'tools_used': ['check_expense_eligibility',
  'check_receipt_requirement',
  'calculate_reimbursement',
  'check_airfare_class',
  'check_submission_timeliness',
  'check_approval_threshold']}

In [28]:
result_005 = run_reimbursement_agent(claims[4])
result_005

Round 1: 1.63s | prompt=1758 | candidates=140 | thoughts=None | total=1898
Round 1: 4 function call(s)
  Tool: check_expense_eligibility | 0.0000s
  Tool: check_receipt_requirement | 0.0000s
  Tool: calculate_reimbursement | 0.0000s
  Tool: check_submission_timeliness | 0.0000s
Round 2: 1.16s | prompt=2180 | candidates=22 | thoughts=None | total=2202
Round 2: 1 function call(s)
  Tool: check_approval_threshold | 0.0000s
Finalization: 1.72s
Agent completed in 2 Gemini tool-calling round(s). Total Gemini tool-calling time: 2.78s
Tools used: ['check_expense_eligibility', 'check_receipt_requirement', 'calculate_reimbursement', 'check_submission_timeliness', 'check_approval_threshold']


{'claim_id': 'CLM-005',
 'decision': 'MANUAL_REVIEW',
 'approved_amount': 0.0,
 'deducted_amount': 0.0,
 'missing_docs': ['Itemized receipt for meals expense of $220.0'],
 'policy_refs': ['POL-CAT-01',
  'POL-RCT-02',
  'POL-PD-01',
  'POL-TIME-01',
  'POL-APR-01'],
 'confidence': 0.95,
 'explanation': 'The claim is submitted within the allowed timeliness window and the meals category is eligible. However, the $220.0 meals expense lacks the required receipt, which triggers a MANUAL_REVIEW condition per policy POL-RCT-02. Per instructions, for MANUAL_REVIEW we do not treat potentially reimbursable amounts as already approved.',
 'tools_used': ['check_expense_eligibility',
  'check_receipt_requirement',
  'calculate_reimbursement',
  'check_submission_timeliness',
  'check_approval_threshold']}

## 10 Results Dashboard

This data-driven summary is rendered from the five agent result objects below. It
shows each decision, approved and deducted amounts, missing documents, and tool
usage. A manual-review fallback is visible when Gemini credentials are unavailable.

In [32]:
from html import escape
from IPython.display import HTML, Markdown, display

all_results = [result_001, result_002, result_003, result_004, result_005]
decision_counts = Counter(result["decision"] for result in all_results)
total_claimed = sum(claim["total_claimed"] for claim in claims)
total_approved = sum(result["approved_amount"] for result in all_results)
total_deducted = sum(result["deducted_amount"] for result in all_results)
max_count = max(decision_counts.values(), default=1)

def render_decision_bar(decision: str, count: int) -> str:
    decision_labels = {
        "APPROVE": ("Approved", "approved"),
        "PARTIAL_APPROVE": ("Partially approved", "partially approved"),
        "REJECT": ("Rejected", "rejected"),
        "MANUAL_REVIEW": ("Manual review", "require manual review"),
    }
    label, outcome_phrase = decision_labels[decision]
    claim_word = "claim" if count == 1 else "claims"
    count_label = f"{count} {claim_word} {outcome_phrase}"
    width_percent = count / max_count * 100
    return (
        f'<div class="bar-row"><strong>{label}</strong>'
        f'<div class="track"><div class="fill" role="img" '
        f'aria-label="{count_label}" title="{count_label}" '
        f'style="width:{width_percent:.0f}%"></div></div>'
        f'<span class="bar-count">{count_label}</span></div>'
    )

decision_bars = "".join(
    render_decision_bar(decision, count)
    for decision, count in sorted(decision_counts.items())
)
assert len(all_results) == len(claims) == 5, "Dashboard expects results for all five provided claims."
markdown_rows = []
for result in all_results:
    missing_docs = "; ".join(result["missing_docs"]) or "None"
    missing_docs = missing_docs.replace("|", "\\|").replace("\n", "<br>")
    markdown_rows.append(
        f"| {result['claim_id']} | {result['decision'].replace('_', ' ')} | "
        f"${result['approved_amount']:,.2f} | ${result['deducted_amount']:,.2f} | "
        f"{missing_docs} | {len(result['tools_used'])} tools |"
    )
markdown_table = "\n".join([
    "| Claim | Decision | Approved | Deducted | Missing documents | Tools |",
    "| :-- | :-- | --: | --: | :-- | --: |",
    *markdown_rows,
])
dashboard_html = f"""
<style>
.dashboard{{font-family:Arial,sans-serif;color:#111827!important;background:#ffffff!important;padding:16px;border-radius:12px;opacity:1!important}}
.dashboard h3,.dashboard p,.bar-row strong,.bar-count{{color:#111827!important;opacity:1!important}}
.dashboard h3{{font-size:16px;font-weight:700}}
.dashboard p{{font-size:14px;line-height:1.5;font-weight:500}}
.bar-row strong,.bar-count{{font-size:14px;font-weight:700}}
.cards{{display:flex;gap:12px;flex-wrap:wrap;margin:14px 0}}
.card{{background:#f1f5f9;border-radius:10px;padding:14px 18px;min-width:145px}}
.label{{font-size:12px;color:#526176}} .value{{font-size:22px;font-weight:700;margin-top:5px}}
.bar-row{{display:grid;grid-template-columns:155px 1fr 220px;gap:10px;align-items:center;margin:8px 0}}
.track{{height:12px;background:#cbd5e1;border-radius:10px;overflow:hidden}}
.fill{{height:100%;background:#f97316;border-radius:10px;display:flex;align-items:center;justify-content:center;color:white;font-size:11px;font-weight:700}}
.dashboard table{{border-collapse:collapse;width:100%;margin-top:14px;font-size:13px}}
.dashboard th,.dashboard td{{padding:9px;border-bottom:1px solid #e2e8f0;text-align:left;vertical-align:top}}
.dashboard th{{background:#f8fafc}}
</style>
<div class="dashboard">
<div class="cards">
<div class="card"><div class="label">Claims reviewed</div><div class="value">{len(all_results)}</div></div>
<div class="card"><div class="label">Total claimed</div><div class="value">${total_claimed:,.2f}</div></div>
<div class="card"><div class="label">Approved</div><div class="value">${total_approved:,.2f}</div></div>
<div class="card"><div class="label">Deducted</div><div class="value">${total_deducted:,.2f}</div></div>
</div>
<h3>Decision breakdown by claim count</h3>
<p>Bar length represents the number of claims for each outcome. The label beside each bar gives both the count and outcome (for example, “2 claims require manual review”); bar lengths are scaled to the largest count.</p>
{decision_bars}
<p>Claim-level results are listed in the data table immediately below.</p></div>
"""
display(HTML(dashboard_html))
display(Markdown(markdown_table))

| Claim | Decision | Approved | Deducted | Missing documents | Tools |
| :-- | :-- | --: | --: | :-- | --: |
| CLM-001 | APPROVE | $1,110.00 | $0.00 | None | 6 tools |
| CLM-002 | REJECT | $0.00 | $380.00 | None | 5 tools |
| CLM-003 | PARTIAL APPROVE | $840.00 | $100.00 | None | 6 tools |
| CLM-004 | MANUAL REVIEW | $0.00 | $0.00 | lodging receipt | 6 tools |
| CLM-005 | MANUAL REVIEW | $0.00 | $0.00 | Itemized receipt for meals expense of $220.0 | 5 tools |

## 11 - Design Notes & Reasoning

- **Policy grounding:** Appendix A rules are represented by stable `POL-*` identifiers and deterministic tools. The model may orchestrate checks and explain results, but tool results are authoritative for policy calculations.
- **Agent flow:** Round 1 requests independent checks for every expense plus claim timeliness and airfare class where relevant. Round 2 always evaluates the approval threshold from the calculated reimbursable total, including when another review trigger already exists.
- **Decision precedence:** Missing required receipts, non-economy airfare, late submission, unknown categories, and amounts above $2,000 route to `MANUAL_REVIEW`. Ineligible items are deducted; capped eligible expenses can result in `PARTIAL_APPROVE`. A review decision does not claim a pending amount is already approved.
- **Assumptions:** Lodging quantities come from the stated nights; meal quantities use the stated trip days where the claim describes daily meals; categories without a per-unit limit use one unit. The supplied claim descriptions and receipt flags are the available evidence.
- **Trade-offs and limitations:** Rules and inputs are local and intentionally small. Receipt contents, duplicate submissions, business-purpose proof, and category-level exceptions are not independently verified. The model can be quota-limited or return malformed output; the notebook safely falls back to review for missing credentials or quota conditions.
- **Next improvements:** Add a duplicate-claim tool, stronger schema validation and retries, and mocked receipt metadata or an approval audit trail if those inputs are supplied.

## 12 - README / Setup

This notebook is the complete assignment deliverable. Use Python 3.10+ with Jupyter and install the dependencies with `pip install jupyter google-genai python-dotenv`. Put `GEMINI_API_KEY=...` in a local `.env` file or export it in the environment before starting the notebook; the key is never stored in notebook source. Keep `.env` private and out of the submitted repository. Run all cells from top to bottom. With no key, the notebook still runs and routes each attempted evaluation to `MANUAL_REVIEW`; set the key and rerun the agent cells to obtain Gemini decisions.

The five generated example outcomes are shown in the agent result cells, the dashboard is rendered inline, and the final cell prints the required JSON array. Save or export the executed notebook to provide demo evidence. Upload the single notebook to a working GitHub repository and submit its shareable link as requested by the assignment.

## 13 - Deliverables & Validation

- **Notebook:** Submit this single notebook as `rahul_maurya.ipynb` in a working GitHub repository and include its shareable link.
- **Results:** The final code cell validates the five claim IDs, exact output keys, allowed decisions, nonnegative amount values, and confidence range before printing the JSON array.
- **Evidence:** The five result cells show generated decisions and explanations; the inline dashboard summarizes those same outcomes. Save the executed notebook or a dashboard screenshot as interview/demo evidence.
- **Credential check:** The API key is read from `.env` or the environment. Do not commit `.env` or include the key in shared outputs.
- **Validation status:** Notebook source syntax and the five saved outputs are verified. A fresh live Gemini run requires a valid key and available API quota.

In [34]:
# Final assignment output: one validated JSON object for each Appendix B claim.
required_fields = {
    "claim_id", "decision", "approved_amount", "deducted_amount",
    "missing_docs", "policy_refs", "confidence", "explanation", "tools_used",
}
assert len(all_results) == 5
assert [result["claim_id"] for result in all_results] == [claim["claim_id"] for claim in claims]
for result in all_results:
    assert set(result) == required_fields
    assert result["decision"] in DECISIONS
    assert isinstance(result["approved_amount"], (int, float)) and result["approved_amount"] >= 0
    assert isinstance(result["deducted_amount"], (int, float)) and result["deducted_amount"] >= 0
    assert isinstance(result["missing_docs"], list)
    assert isinstance(result["policy_refs"], list)
    assert isinstance(result["tools_used"], list)
    assert isinstance(result["confidence"], (int, float)) and 0 <= result["confidence"] <= 1
    assert isinstance(result["explanation"], str) and result["explanation"].strip()

print(json.dumps(all_results, indent=2))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-CAT-01",
      "POL-RCT-01",
      "POL-PD-02",
      "POL-PD-01",
      "POL-TIME-01",
      "POL-AIR-01",
      "POL-APR-02"
    ],
    "confidence": 1.0,
    "explanation": "All expenses are eligible, receipts are attached, submission is timely, airfare is economy, and the amount has been verified and approved under manager approval threshold policy.",
    "tools_used": [
      "check_expense_eligibility",
      "check_receipt_requirement",
      "calculate_reimbursement",
      "check_submission_timeliness",
      "check_airfare_class",
      "check_approval_threshold"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-CAT-02",
      "POL-RCT-01",
      "POL-TIME-01",
      "POL-AP